# Meilenstein 2 – Modellierung und Versuchsaufbau

**Kurs:** Einführung in die Energieinformatik  
**Simulationstool:** Sim-PowerCS (Java)  
**Datum:** 2026-06-01

---
## 1. Präzisierung der Fragestellung

> **Ab welcher Batteriekapazität und unter welchen Preisbedingungen können prioritätsbasierte Gerätesteuerungsstrategien die Energiebezugskosten eines PV-Heimspeichersystems gegenüber einer reaktiven Basisstrategie signifikant senken – und welche Rückkopplungseffekte zwischen Gerätenutzung, Batteriezustand und Preisverlauf sind hier ausschlaggebend?**

Die Fragestellung bezieht sich auf das dynamische Verhalten eines lokalen Energiesystems mit PV und Speicher. Sie ist mit Sim-PowerCS untersuchbar, da das Tool sowohl verschiedene Service-Controller als auch die Batteriekapazität und Strompreise konfigurierbar macht. Durch gezielte Parametervariationen (Kapazität, Tarif, Steuerungsstrategie) kann die Frage systematisch analysiert werden.

---
## 2. Beschreibung des Simulationsmodells

### Systemkomponenten

| Komponente | Rolle im System | Sim-PowerCS-Klasse |
|------------|----------------|--------------------|
| PV-Anlage | Erzeugung – speist je nach Tageszeit Energie ins System | `ModeledPhotovoltaicModule` |
| Batterie | Speicher – puffert Überschuss, versorgt bei Bedarf | `BasicBattery` |
| Stromnetz | Netzanschluss – Bezug wenn Speicher leer, Einspeisung bei Überschuss | `BasicGrid` |
| Passive Last | Grundverbrauch – nicht steuerbar (Kühlschrank etc.) | `ModeledPasiveLoad` |
| Smart Services | Steuerbare Geräte mit konfigurierbarer Priorität und Zeitfenster | `ServicesActiveLoad` |
| Service-Controller | Entscheidet welche Smart Services ein- oder ausgeschaltet werden | `BasicController` / `PriorityBasedController` |
| Energie-Controller | Entscheidet ob Batterie oder Netz bei Bedarf genutzt wird (Batterie hat Vorrang) | `BasicEnergyController` |

### Systemzustände (verändern sich über die Zeit)

- **Batterieladestand (%)** – zentraler Zustand, beeinflusst Steuerungsentscheidungen
- **PV-Erzeugung (Wm)** – tageszeit- und modellabhängig, nicht steuerbar
- **Gesamtlast (Wm)** – Summe aus passiver Grundlast und aktiven Smart Services
- **Netzbezug / -einspeisung (Wm)** – ergibt sich aus Differenz Erzeugung–Verbrauch nach Batteriepuffer
- **Kumulierte Kosten (€)** – akkumulierter Netzbezug bewertet mit stündlichem Marktpreis
- **Gerätezustände (ein/aus)** – je Smart Service, gesteuert durch den Service-Controller

### Zeitliche Auflösung

- Simulationsschritt: **1 Minute**
- Simulationsdauer: **7 Tage** (10.080 Schritte)
- Strompreise: stündlich aktualisiert (24 Stufen/Tag aus `EnergyPrices.csv`)

---
## 3. Systemmodell und Szenarien

### Steuerungs- und Kontrolllogik

Das System verwendet zwei unabhängige Controller:

**BasicEnergyController** (fest, in allen Szenarien gleich):  
Bei Energiebedarf wird zuerst die Batterie genutzt, erst wenn diese nicht ausreicht, wird Strom vom Netz bezogen. Bei Überschuss wird die Batterie geladen; ist sie voll, wird ins Netz eingespeist.

**Service-Controller** (variiert je Szenario):  

| Controller | Logik | Einsatz |
|------------|-------|---------|
| `BasicController` | Schaltet alle Smart Services ein, sobald Batterie = 100 % und PV-Überschuss vorhanden. Sonst alle aus. Kein Prioritätsbewusstsein. | Referenzszenario |
| `PriorityBasedController` | Schaltet niedrig priorisierte Geräte präventiv ab, sobald der gleitende Batterie-Durchschnitt eine untere Schwelle (20 %) unterschreitet. Hochpriorisierte Geräte bleiben länger aktiv. | Variationsszenarien |

Die fünf steuerbaren Geräte (`services.xml`) in aufsteigender Priorität (niedrig = zuerst abgeschaltet):

| Gerät | Leistung | Priorität | Betriebsfenster |
|-------|---------|-----------|----------------|
| Fuente de agua | 35 W | 1 | 09–15 Uhr |
| StreamServices | 30 W | 2 | ganztags |
| Motor piscina | 600 W | 4 | 09–17 Uhr |
| Internet | 40 W | 8 | ganztags |
| Videograbador | 20 W | 10 | ganztags |

---

### Referenzszenario

Das Referenzszenario bildet die Ausgangsbasis für alle Vergleiche.

| Parameter | Wert |
|-----------|------|
| Controller | `BasicController` |
| Batteriekapazität | 4.000 Wh |
| Max. Ladeleistung | 2.500 W |
| Strompreismodell | dynamisch (EnergyPrices.csv, Spreizung ca. 3,5×) |
| Simulationsdauer | 7 Tage |

---

### Variationsszenarien

Pro Variation wird **nur ein Parameter** gegenüber dem Referenzszenario geändert. Jede Variation wird mit beiden Controllern (Basic + Priority) durchgeführt, um den Effekt der Steuerungsstrategie isoliert zu messen.

**Variation 1 – Controller (Teilfrage T1)**  
Tauscht `BasicController` gegen `PriorityBasedController` bei sonst identischer Konfiguration.  
*Begründung:* Direkter Strategievergleich als Grundlage für alle weiteren Aussagen.

**Variation 2 – Batteriekapazität (Teilfrage T2)**  
Kapazitäten: 1 kWh / 2 kWh / 4 kWh (Referenz) / 6 kWh / 10 kWh, je mit Basic und Priority.  
*Begründung:* Der Puffer bestimmt, wie lange die Strategie Netzstrom vermeiden kann. Unterhalb einer Schwelle hat kein Controller Spielraum; oberhalb braucht auch Basic selten Netzstrom.

**Variation 3 – Stromtarif (Teilfrage T3)**  
Flachtarif: alle 24 Stunden auf den Durchschnittswert (0,161 €/kWh) setzen, je mit Basic und Priority.  
*Begründung:* Die Zeitstruktur des Preises ist das entscheidende Element. Ohne Spreizung verliert der indirekte Preisschutz durch Priority seinen Wert.

---

### Mess- und Auswertungsgrößen

| Größe | Art | Wozu |
|-------|-----|------|
| Nettobezugskosten (€) | Summe (Buy − Sell über 7 Tage) | Hauptmetrik, Szenariovergleich |
| Batterieladestand (%) | Zeitlicher Verlauf | Rückkopplungsanalyse: wann greift Priority? |
| Netzbezug (Wh) | Zeitlicher Verlauf | Zeigt zu welchen Preiszeiten Strom gekauft wird |
| Gerätenutzung je Service (%) | relativer Anteil | Zeigt Komfortauswirkungen der Priorisierung |
| PV-Erzeugung vs. Last | Zeitlicher Verlauf | Energiebilanz, Einordnung der Systemdynamik |